# Spatial Detector Comparison

Compares on **one Pavia-U scenario**:
- **DSM** (our global per-pixel score)
- **Spatial DSM** — CF-Attn, CF-Attn-CFAR, NeighborMLP
- **AMF** global + **AMF-local** (k×k window SCM)
- **CEM** global + **CEM-local** (k×k window SCM)
- **GMM-Levin**

**Metrics:** pAUC@0.05, AUC, Pd@Pfa=0.05, per-class Pfa + Pfa_avg.
**Outputs:** false-color, label map, detection maps, ROC, per-class Pfa bars, summary table.

Deep nets train on **GPU** (set Runtime → Change runtime type → GPU).

In [ ]:
# ── Cell 1: Clone repo (pinned branch) + install deps ─────────────
import os, sys

REPO_URL      = 'https://github.com/michaelpiro/final-paper-experiment.git'
LOCAL_PROJECT = '/content/proj'
BRANCH        = 'iid-unified-experiment'   # spatial compare code lives here

if os.path.exists(os.path.join(LOCAL_PROJECT, '.git')):
    !git -C {LOCAL_PROJECT} fetch --all -q
    !git -C {LOCAL_PROJECT} checkout {BRANCH} -q
    !git -C {LOCAL_PROJECT} pull origin {BRANCH} -q
else:
    !git clone -b {BRANCH} --depth 1 {REPO_URL} {LOCAL_PROJECT}

!pip install -q scikit-learn scipy tqdm matplotlib pyyaml

for p in [LOCAL_PROJECT, os.path.join(LOCAL_PROJECT, 'experiments', 'spatial')]:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(LOCAL_PROJECT)
print('Ready. CWD:', os.getcwd())
!git -C {LOCAL_PROJECT} log --oneline -1

In [ ]:
# ── Cell 2: GPU check ─────────────────────────────────────
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU — enable Runtime > Change runtime type > GPU for fast deep nets.')
print('PyTorch', torch.__version__, ' device =', DEVICE)

In [ ]:
# ── Cell 3: Mount Drive + link pavia-u.mat ──────────────────────
import os
RESULTS = '/content/results'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS = '/content/drive/MyDrive/final_paper/compare_results'
    os.makedirs('/content/proj/data', exist_ok=True)
    DST = '/content/proj/data/pavia-u.mat'
    if not os.path.exists(DST):
        for src in ['/content/drive/MyDrive/final_paper/pavia-u.mat',
                    '/content/drive/MyDrive/final_paper/real_datasets/pavia-u.mat']:
            if os.path.exists(src):
                os.symlink(src, DST); print('Linked dataset from', src); break
        else:
            print('WARNING: pavia-u.mat not found on Drive.')
    else:
        print('Dataset already linked.')
except Exception as e:
    print('Drive not mounted:', repr(e))
os.makedirs(RESULTS, exist_ok=True)
assert os.path.exists('/content/proj/data/pavia-u.mat'), 'pavia-u.mat missing!'
print('RESULTS dir:', RESULTS)

In [ ]:
# ── Cell 4: Settings ────────────────────────────────────
# SCENARIO: 0-3 = manual boxes, 4-7 = random boxes
SCENARIO = 0
QUICK    = False   # True = fast smoke test (few epochs)
print(f'scenario={SCENARIO}  quick={QUICK}')

In [ ]:
# ── Cell 5: Run the comparison ─────────────────────────────
import subprocess, sys
cmd = [sys.executable, '-u', 'experiments/spatial/run_compare.py',
       '--config', 'experiments/spatial/colab.yaml',
       '--results_dir', RESULTS,
       '--scenario', str(SCENARIO)]
if QUICK:
    cmd.append('--dry-run')
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# ── Cell 6: Show summary table ─────────────────────────────
import glob, os
import pandas as pd
from IPython.display import display, Markdown

run_dir = sorted(glob.glob(os.path.join(RESULTS, 'compare_*')))[-1]
print('Run:', run_dir)
df = pd.read_csv(os.path.join(run_dir, 'summary_table.csv'))
display(df.style.background_gradient(subset=['pAUC@0.05','AUC','Pd@Pfa=0.05'], cmap='Greens')
                .background_gradient(subset=['Pfa_avg','Pfa_max'], cmap='Reds')
                .format({c: '{:.3f}' for c in df.columns if c != 'Detector'}))

In [ ]:
# ── Cell 7: Display all figures inline ───────────────────────
import glob, os
import matplotlib.pyplot as plt, matplotlib.image as mpimg

order = ['false_color', 'label_map', 'detection_maps', 'roc', 'pfa_per_class']
for name in order:
    png = os.path.join(run_dir, name + '.png')
    if os.path.exists(png):
        img = mpimg.imread(png)
        fig, ax = plt.subplots(figsize=(13, 9))
        ax.imshow(img); ax.axis('off'); ax.set_title(name, fontsize=10)
        plt.tight_layout(); plt.show()